# 🧩 Mini-Lab: Baseline, then a 50-Row Gold Set

**Module 9: Is It Any Good? Quality, Latency, Cost** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Run** a majority-class / keyword baseline and record its score
2. **Verify** a 50-row gold set
3. **Compute** accuracy, per-class precision/recall, and a confusion matrix

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Baseline comparison | Judge output against a simple reference, not against "looks good" |
| Gold set | Small verified labelled sample; state the sample-size caveat |
| Precision and recall | Of the flagged versus of the existing; per-class beats headline |


---
## Baseline first

Score the dumbest thing that could work before you score the model.


## Why Start With a Baseline?

Imagine your AI classifier scores **82% accuracy** on a support-ticket dataset. Impressive? Not necessarily.

If **80% of the tickets are "billing"** issues, a system that *always* guesses "billing" — without reading the ticket at all — would also hit 80%. Your fancy model only beats the dumbest possible approach by 2 percentage points.

A **majority-class baseline** is the simplest baseline: always predict the most frequent label. If your AI can't beat this, it isn't adding value.

## Step 1 — Create a Small Labeled Dataset

We'll simulate a set of support tickets with human-verified labels (a "gold set") and the predictions an AI system made.

In [1]:
# Simulated gold-set: true labels and AI predictions for 20 support tickets
true_labels = [
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "billing", "billing",           # 14 billing
    "technical", "technical", "technical",                  # 3 technical
    "account", "account", "account",                       # 3 account
]

ai_predictions = [
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "technical", "account",          # 2 wrong on billing
    "technical", "technical", "billing",                   # 1 wrong on technical
    "account", "account", "billing",                      # 1 wrong on account
]

print(f"Total tickets: {len(true_labels)}")
print(f"Label distribution: billing={true_labels.count('billing')}, "
      f"technical={true_labels.count('technical')}, "
      f"account={true_labels.count('account')}")

Total tickets: 20
Label distribution: billing=14, technical=3, account=3


## Step 2 — Build the Majority-Class Baseline

Find the most common label, then create a "prediction" list that always guesses that label.

In [2]:
from collections import Counter

# Find the majority class
label_counts = Counter(true_labels)
majority_class = label_counts.most_common(1)[0][0]

print(f"Majority class: '{majority_class}' ({label_counts[majority_class]}/{len(true_labels)} tickets)")

# The baseline always predicts the majority class
baseline_predictions = [majority_class] * len(true_labels)

Majority class: 'billing' (14/20 tickets)


## Step 3 — Compare Accuracy: Baseline vs. AI

In [3]:
def accuracy(true, predicted):
    """Fraction of predictions that match the true labels."""
    correct = sum(t == p for t, p in zip(true, predicted))
    return correct / len(true)

baseline_accuracy = accuracy(true_labels, baseline_predictions) * 100
ai_accuracy       = accuracy(true_labels, ai_predictions) * 100
margin             = ai_accuracy - baseline_accuracy

print(f"Baseline accuracy (always guess '{majority_class}'): {baseline_accuracy:.1f}%")
print(f"AI system accuracy:                                  {ai_accuracy:.1f}%")
print(f"Margin over baseline:                                {margin:+.1f} percentage points")
print()

if margin > 0:
    print("✅ The AI system beats the majority-class baseline.")
else:
    print("❌ The AI system does NOT beat the baseline — it adds no value.")

Baseline accuracy (always guess 'billing'): 70.0%
AI system accuracy:                                  80.0%
Margin over baseline:                                +10.0 percentage points

✅ The AI system beats the majority-class baseline.


## Step 4 — Interpret the Result

The AI scored higher than the baseline — but look at the margin. With a skewed dataset (70% billing), the baseline is already at 70%. A small margin means the AI is barely doing better than guessing the same thing every time.

**The takeaway:** always report your AI's accuracy *relative to the baseline*, not in isolation. An 80% accuracy sounds great until you learn the baseline is 78%.

In [4]:
from IPython.display import display, Markdown
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

report = f"""
### Baseline Evaluation Report

| Metric | Value |
|--------|-------|
| Dataset size | {len(true_labels)} tickets |
| Majority class | `{majority_class}` ({label_counts[majority_class]}/{len(true_labels)}) |
| Baseline accuracy | {baseline_accuracy:.1f}% |
| AI accuracy | {ai_accuracy:.1f}% |
| **Margin** | **{margin:+.1f} pp** |

> **Rule of thumb:** If the margin is small, invest effort in improving the
> AI system before shipping — or question whether you need AI at all for this task.
"""

md(report)


### Baseline Evaluation Report

| Metric | Value |
|--------|-------|
| Dataset size | 20 tickets |
| Majority class | `billing` (14/20) |
| Baseline accuracy | 70.0% |
| AI accuracy | 80.0% |
| **Margin** | **+10.0 pp** |

> **Rule of thumb:** If the margin is small, invest effort in improving the
> AI system before shipping — or question whether you need AI at all for this task.


## What to watch

The next cell is a **gold set** — a small, human-verified labeled sample. Watch the size (30 tickets) when you read accuracy: predict whether one headline number will hide a class the model misses.


## Step 1 — Build a Gold Set

We'll reuse the same support-ticket scenario from the baseline mini-lab, but expand it to 30 tickets so we have enough examples per class to compute meaningful per-class metrics.

In [1]:
# Gold set: 30 support tickets with human-verified labels and AI predictions
gold_labels = [
    # 15 billing
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "billing", "billing", "billing",
    # 9 technical
    "technical", "technical", "technical", "technical", "technical",
    "technical", "technical", "technical", "technical",
    # 6 account
    "account", "account", "account", "account", "account", "account",
]

ai_predictions = [
    # predictions for billing tickets (15): 12 correct, 2→technical, 1→account
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "technical", "technical", "account",
    # predictions for technical tickets (9): 6 correct, 2→billing, 1→account
    "technical", "technical", "technical", "technical", "technical",
    "technical", "billing", "billing", "account",
    # predictions for account tickets (6): 4 correct, 1→billing, 1→technical
    "account", "account", "account", "account", "billing", "technical",
]

from collections import Counter
dist = Counter(gold_labels)
print(f"Gold set size: {len(gold_labels)} tickets")
print(f"Label distribution: {dict(dist)}")
print(f"\n⚠️  Sample-size caveat: with only {len(gold_labels)} rows, treat all "
      f"metrics as directional estimates, not exact measurements.")

Gold set size: 30 tickets
Label distribution: {'billing': 15, 'technical': 9, 'account': 6}

⚠️  Sample-size caveat: with only 30 rows, treat all metrics as directional estimates, not exact measurements.


## Step 2 — Compute Overall Accuracy

Accuracy is the simplest metric: what fraction of predictions matched the gold labels? Recall from the prior mini-lab that accuracy alone can be misleading — we need to compare it against the majority-class baseline.

In [2]:
def accuracy(true, predicted):
    """Fraction of predictions that match the true labels."""
    correct = sum(t == p for t, p in zip(true, predicted))
    return correct / len(true)

# AI accuracy
ai_acc = accuracy(gold_labels, ai_predictions) * 100

# Majority-class baseline (from prior mini-lab concept)
majority_class = Counter(gold_labels).most_common(1)[0][0]
baseline_acc = accuracy(gold_labels, [majority_class] * len(gold_labels)) * 100

margin = ai_acc - baseline_acc

print(f"Majority-class baseline (always '{majority_class}'): {baseline_acc:.1f}%")
print(f"AI accuracy:                                         {ai_acc:.1f}%")
print(f"Margin over baseline:                                {margin:+.1f} pp")

Majority-class baseline (always 'billing'): 50.0%
AI accuracy:                                         73.3%
Margin over baseline:                                +23.3 pp


## Step 3 — Build a Confusion Matrix

Accuracy tells you *how many* errors, but not *what kind*. A **confusion matrix** shows exactly where the AI gets confused: which true class gets mislabeled as which predicted class.

Each row is a **true** label; each column is what the AI **predicted**. Correct predictions sit on the diagonal.

In [3]:
def confusion_matrix(true, predicted, labels):
    """Build a confusion matrix as a dict-of-dicts: matrix[true][predicted] = count."""
    matrix = {t: {p: 0 for p in labels} for t in labels}
    for t, p in zip(true, predicted):
        matrix[t][p] += 1
    return matrix

labels = sorted(set(gold_labels))  # ['account', 'billing', 'technical']
cm = confusion_matrix(gold_labels, ai_predictions, labels)

# Display as a readable table
from IPython.display import display, Markdown
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

header = "| True \\ Predicted | " + " | ".join(labels) + " |"
sep    = "|" + "---|" * (len(labels) + 1)
rows = []
for t in labels:
    vals = " | ".join(str(cm[t][p]) for p in labels)
    rows.append(f"| **{t}** | {vals} |")

md("### Confusion Matrix\n\n" + header + "\n" + sep + "\n" + "\n".join(rows) + "\n\n"
   "*Rows = true label, Columns = AI prediction. Diagonal = correct.*")

### Confusion Matrix

| True \ Predicted | account | billing | technical |
|---|---|---|---|
| **account** | 4 | 1 | 1 |
| **billing** | 1 | 12 | 2 |
| **technical** | 1 | 2 | 6 |

*Rows = true label, Columns = AI prediction. Diagonal = correct.*

## Step 4 — Compute Per-Class Precision and Recall

Watch the next output for per-class **precision** and **recall**, and the confusion matrix behind them. Predict which class the model flags too often versus which it misses.


In [4]:
def precision_recall(cm, labels):
    """Compute per-class precision and recall from a confusion matrix."""
    metrics = {}
    for cls in labels:
        tp = cm[cls][cls]  # true positives: correctly predicted as cls

        # Precision denominator: everything the AI predicted as cls
        predicted_as_cls = sum(cm[t][cls] for t in labels)
        precision = tp / predicted_as_cls if predicted_as_cls > 0 else 0.0

        # Recall denominator: everything that truly was cls
        actually_cls = sum(cm[cls][p] for p in labels)
        recall = tp / actually_cls if actually_cls > 0 else 0.0

        metrics[cls] = {"precision": precision, "recall": recall, "support": actually_cls}
    return metrics

metrics = precision_recall(cm, labels)

# Display as a table
rows = ["| Class | Precision | Recall | Support |", "|---|---|---|---|"]
for cls in labels:
    m = metrics[cls]
    rows.append(f"| **{cls}** | {m['precision']:.0%} | {m['recall']:.0%} | {m['support']} |")

md("### Per-Class Precision & Recall\n\n" + "\n".join(rows) + "\n\n"
   "*Support = number of true examples in the gold set for that class.*")

### Per-Class Precision & Recall

| Class | Precision | Recall | Support |
|---|---|---|---|
| **account** | 67% | 67% | 6 |
| **billing** | 80% | 80% | 15 |
| **technical** | 67% | 67% | 9 |

*Support = number of true examples in the gold set for that class.*

## Step 5 — Interpret the Results

Let's put everything together into a single evaluation report that an engineer or stakeholder could read.

In [5]:
report = f"""### 📋 Gold-Set Evaluation Report

| Metric | Value |
|--------|-------|
| Gold set size | {len(gold_labels)} tickets |
| Classes | {', '.join(labels)} |
| Majority-class baseline | {baseline_acc:.1f}% |
| AI accuracy | {ai_acc:.1f}% |
| Margin over baseline | {margin:+.1f} pp |

#### Per-Class Breakdown

| Class | Precision | Recall | Support |
|-------|-----------|--------|--------|
"""

for cls in labels:
    m = metrics[cls]
    report += f"| {cls} | {m['precision']:.0%} | {m['recall']:.0%} | {m['support']} |\n"

report += f"""
#### Key Observations

- **Billing** has the highest recall ({metrics['billing']['recall']:.0%}) — the AI rarely misses billing tickets.
- **Technical** has lower recall ({metrics['technical']['recall']:.0%}) — {9 - cm['technical']['technical']} out of {metrics['technical']['support']} technical tickets were misclassified.
- **Account** is the smallest class ({metrics['account']['support']} examples) — metrics for small classes are noisy.

> ⚠️ **Sample-size caveat:** This gold set has only {len(gold_labels)} rows.
> All metrics above are estimates. Before making a ship decision,
> consider expanding the gold set to 50–100+ rows for tighter confidence.
"""

md(report)

### 📋 Gold-Set Evaluation Report

| Metric | Value |
|--------|-------|
| Gold set size | 30 tickets |
| Classes | account, billing, technical |
| Majority-class baseline | 50.0% |
| AI accuracy | 73.3% |
| Margin over baseline | +23.3 pp |

#### Per-Class Breakdown

| Class | Precision | Recall | Support |
|-------|-----------|--------|--------|
| account | 67% | 67% | 6 |
| billing | 80% | 80% | 15 |
| technical | 67% | 67% | 9 |

#### Key Observations

- **Billing** has the highest recall (80%) — the AI rarely misses billing tickets.
- **Technical** has lower recall (67%) — 3 out of 9 technical tickets were misclassified.
- **Account** is the smallest class (6 examples) — metrics for small classes are noisy.

> ⚠️ **Sample-size caveat:** This gold set has only 30 rows.
> All metrics above are estimates. Before making a ship decision,
> consider expanding the gold set to 50–100+ rows for tighter confidence.


## 🎯 Summary

### Key Takeaways

1. **Gold set** — a small, human-verified labeled dataset that serves as ground truth; always state its size alongside any metric you report
2. **Confusion matrix** — shows exactly which classes get confused with which, going beyond a single accuracy number
3. **Precision** — of everything the AI predicted as class X, how many were actually X (measures false-positive rate)
4. **Recall** — of everything that truly was class X, how many did the AI catch (measures false-negative rate)
5. **Baseline comparison** — accuracy only means something when compared to the majority-class baseline; always report the margin